# 第 3 章 · warp 与 SIMT：32 个线程一起走，分岔了怎么办

本章回答：block 里的线程真的是各跑各的吗？什么是 warp？`if` 语句让一个 warp 里的线程走了不同的路，会发生什么、慢多少？为什么 block 大小要取 32 的倍数？

配套讲义：仓库里 `cuda-programming/notes/lessons/03-warp-simt.md`（讲解、动画、练习都在讲义里）。对应原文：PG §1.2.2.2 Warps and SIMT（`materials/text/cuda-programming-guide.txt` 第 1077–1171 行）、§3.2.2.1 SIMT Execution Model 与 Independent Thread Scheduling（第 8645–8715 行）；Oxford lec3（`text/oxford-lec3.txt`，控制流与同步）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU：代码用 NumPy 在 CPU 上模拟 GPU 的规则（线程编号、32 字节一箱、32 个 bank……）。

In [ ]:
import math

import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 按 lane 奇偶分 | 两条分支各 10 | `block_time(32, lambda tid: tid % 2 == 0, 10, 10)` | [20]，lane 效率 50% |
| 第二步 按 warp 分 | `(tid / 32) % 2` | `block_time(64, lambda tid: (tid // 32) % 2 == 0, 10, 10)` | [10, 10]，没有浪费 |
| 第二步 常见写法 | 哪些 warp 分岔 | `diverged_warps(64, cond)` | `tid % 2`：[0, 1]；`tid < 16`：[0]；`i < 40`：[1] |
| 第三步 100 个线程 | $\lceil 100 / 32 \rceil$ | `lane_utilization(100)` | 4 个 warp，最后一个 4 人，78% |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**常量 `WARP`**（来自 `ch03_warp_divergence.py`）

In [ ]:
WARP = 32

**函数 `warp_time`**（来自 `ch03_warp_divergence.py`）：一个 warp 执行 if (cond) a(); else b(); 的耗时。take_a 是 32 个 lane 的条件真假。

In [ ]:
def warp_time(take_a: np.ndarray, t_a: float, t_b: float) -> float:
    """一个 warp 执行 if (cond) a(); else b(); 的耗时。take_a 是 32 个 lane 的条件真假。

    有 lane 走 a 就执行一遍 a，有 lane 走 b 就执行一遍 b；两边都有人走就是两趟（分岔）。
    """
    return (t_a if take_a.any() else 0) + (t_b if (~take_a).any() else 0)

**函数 `block_time`**（来自 `ch03_warp_divergence.py`）：整个 block 按 32 个一组切成 warp，返回每个 warp 的耗时。cond(tid) 给出每个线程的条件。

In [ ]:
def block_time(block_dim: int, cond, t_a: float, t_b: float) -> list[float]:
    """整个 block 按 32 个一组切成 warp，返回每个 warp 的耗时。cond(tid) 给出每个线程的条件。"""
    tid = np.arange(block_dim)
    take_a = cond(tid)
    return [warp_time(take_a[w * WARP:(w + 1) * WARP], t_a, t_b) for w in range(math.ceil(block_dim / WARP))]

**函数 `diverged_warps`**（来自 `ch03_warp_divergence.py`）：哪些 warp 内部条件不一致（分岔）。

In [ ]:
def diverged_warps(block_dim: int, cond) -> list[int]:
    """哪些 warp 内部条件不一致（分岔）。"""
    take = cond(np.arange(block_dim))
    out = []
    for w in range(math.ceil(block_dim / WARP)):
        part = take[w * WARP:(w + 1) * WARP]
        if part.any() and not part.all():
            out.append(w)
    return out

**函数 `lane_utilization`**（来自 `ch03_warp_divergence.py`）：block 线程数不是 32 的倍数时：（warp 数, 最后一个 warp 的线程数, 算力利用率）。

In [ ]:
def lane_utilization(block_dim: int) -> tuple[int, int, float]:
    """block 线程数不是 32 的倍数时：（warp 数, 最后一个 warp 的线程数, 算力利用率）。"""
    warps = math.ceil(block_dim / WARP)
    last = block_dim - (warps - 1) * WARP
    return warps, last, block_dim / (warps * WARP)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二步 if (threadIdx.x % 2 == 0)：a、b 各耗时 10")
t = block_time(32, lambda tid: tid % 2 == 0, 10, 10)
print(f"  一个 warp 耗时 {t[0]}（没有分岔时只要 10），lane 效率 = {10 / t[0]:.0%}")

In [ ]:
print("\n第二步 改成按 warp 分组：if ((threadIdx.x / 32) % 2 == 0)")
t = block_time(64, lambda tid: (tid // 32) % 2 == 0, 10, 10)
print(f"  warp 0、warp 1 各耗时 {t}：每个 warp 只走一条路，没有浪费")

In [ ]:
print("\n第二步 常见写法，64 个线程的 block 里哪些 warp 分岔")
print("  if (threadIdx.x % 2 == 0):", diverged_warps(64, lambda tid: tid % 2 == 0))
print("  if (threadIdx.x < 16)    :", diverged_warps(64, lambda tid: tid < 16))
print("  if (i < n)，n = 40       :", diverged_warps(64, lambda tid: tid < 40), "（只有最后一个 warp）")

In [ ]:
print("\n第三步 block 有 100 个线程")
warps, last, util = lane_utilization(100)
print(f"  warp 数 = {warps}，最后一个 warp 只有 {last} 个线程，空着 {WARP - last} 个 lane，利用率 = {util:.0%}")

## 3. 代码里才看得到的

1. **分岔的代价按 warp 算，不按线程算**：`warp_time` 只看这 32 个 lane 里"有没有人"走某条路，有一个人走就整组执行一遍。所以同样是一半线程走 a，按 lane 奇偶分和按 warp 分耗时差一倍。
2. **这里的"时间单位"是编的**：讲义设 a、b 各 10 个单位，只为比较相对快慢；真实耗时还要看指令数和延迟。

## 4. 自己改一改

- 把条件换成 `lambda tid: tid % 4 == 0`，看每个 warp 还分不分岔、耗时多少。
- 把 `lane_utilization(100)` 换成 96、97，看最后一个 warp 的空位怎么变。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 32 人一个班，班长喊一个口令，大家做同一个动作（SIMT）；
> 分岔了就分两趟喊，不走这条路的原地等；
> 分岔只在 warp 里，按 warp 分组就不浪费；
> block 取 32 的倍数，别让最后一班坐不满；
> Volta 以后不再保证齐步走，要同步就写 `__syncwarp()`。